# Utfordring: Analysere tekst om Data Science

I dette eksemplet skal vi gjøre en enkel øvelse som dekker alle steg i en tradisjonell data science-prosess. Du trenger ikke å skrive noe kode, du kan bare klikke på cellene under for å kjøre dem og observere resultatet. Som en utfordring oppfordres du til å prøve denne koden med forskjellige data.

## Mål

I denne leksjonen har vi diskutert forskjellige konsepter relatert til Data Science. La oss prøve å oppdage flere relaterte konsepter ved å gjøre litt **tekstutvinning**. Vi begynner med en tekst om Data Science, trekker ut nøkkelord fra den, og prøver deretter å visualisere resultatet.

Som tekst vil jeg bruke siden om Data Science fra Wikipedia:


In [ ]:
url = 'https://en.wikipedia.org/wiki/Data_science'

## Trinn 1: Skaffe dataene

Det første trinnet i enhver datavitenskapsprosess er å skaffe dataene. Vi vil bruke `requests`-biblioteket for å gjøre det:


In [ ]:
import requests

# Define a custom header.
headers = {
    'User-Agent': 'DataScienceChallenge/1.0 (myemail@gmail.com)'
}

# Pass the headers into the get request
response = requests.get(url, headers=headers)

if response.status_code == 200:
    text = response.content.decode('utf-8')
    print(text[:1000])
else:
    print(f"Error: {response.status_code}")

## Trinn 2: Transformere dataene

Neste steg er å konvertere dataene til et format som egner seg for bearbeiding. I vårt tilfelle har vi lastet ned HTML-kildekode fra siden, og vi må konvertere den til ren tekst.

Det finnes mange måter å gjøre dette på. Vi vil bruke [BeautifulSoup](https://www.crummy.com/software/BeautifulSoup/), et populært Python-bibliotek for å analysere HTML. BeautifulSoup lar oss målrette spesifikke HTML-elementer, slik at vi kan fokusere på hovedinnholdet i artikkelen fra Wikipedia og redusere noe navigasjonsmenyer, sidepaneler, bunntekster og annet irrelevante innhold (selv om noe standardtekst fremdeles kan være igjen).


Først må vi installere BeautifulSoup-biblioteket for HTML-parsing:


In [ ]:
import sys
!{sys.executable} -m pip install beautifulsoup4

In [ ]:
from bs4 import BeautifulSoup

# Parse the HTML content
soup = BeautifulSoup(text, 'html.parser')

# Extract only the main article content from Wikipedia
# Wikipedia uses 'mw-parser-output' class for the main article content
content = soup.find('div', class_='mw-parser-output')

def clean_wikipedia_content(content_node):
    """Remove common non-article elements from a Wikipedia content node."""
    # Strip jump links, navboxes, reference lists/superscripts, edit sections, TOC, sidebars, etc.
    selectors = [
        '.mw-jump-link',
        '.navbox',
        '.reflist',
        'sup.reference',
        '.mw-editsection',
        '.hatnote',
        '.metadata',
        '.infobox',
        '#toc',
        '.toc',
        '.sidebar',
    ]
    for selector in selectors:
        for el in content_node.select(selector):
            el.decompose()

if content:
    # Clean the content node to better approximate article text only.
    clean_wikipedia_content(content)
    text = content.get_text(separator=' ', strip=True)
    print(text[:1000])
else:
    print("Could not find main content. Using full page text.")
    text = soup.get_text(separator=' ', strip=True)
    print(text[:1000])

## Steg 3: Skaffe innsikt

Det viktigste steget er å gjøre om dataene våre til en form hvor vi kan trekke innsikter. I vårt tilfelle ønsker vi å hente ut nøkkelord fra teksten, og se hvilke nøkkelord som er mest meningsfulle.

Vi vil bruke Python-biblioteket kalt [RAKE](https://github.com/aneesha/RAKE) for nøkkelordutvinning. Først, la oss installere dette biblioteket hvis det ikke allerede er installert: 


In [ ]:
import sys
!{sys.executable} -m pip install nlp_rake

Hovedfunksjonaliteten er tilgjengelig fra `Rake`-objektet, som vi kan tilpasse ved hjelp av noen parametere. I vårt tilfelle setter vi minimumslengden på et nøkkelord til 5 tegn, minimum frekvens av et nøkkelord i dokumentet til 3, og maksimalt antall ord i et nøkkelord - til 2. Du kan gjerne eksperimentere med andre verdier og observere resultatet.


In [ ]:
import nlp_rake
extractor = nlp_rake.Rake(max_words=2,min_freq=3,min_chars=5)
res = extractor.apply(text)
res


Vi fikk en liste over termer sammen med tilhørende grad av viktighet. Som du kan se, er de mest relevante disiplinene, som maskinlæring og big data, til stede i listen på topposisjoner.

## Trinn 4: Visualisere resultatet

Folk kan tolke data best i visuell form. Derfor gir det ofte mening å visualisere dataene for å trekke noen innsikter. Vi kan bruke `matplotlib`-biblioteket i Python for å plotte enkel fordeling av nøkkelordene med deres relevans:


In [ ]:
import matplotlib.pyplot as plt

def plot(pair_list):
    k,v = zip(*pair_list)
    plt.bar(range(len(k)),v)
    plt.xticks(range(len(k)),k,rotation='vertical')
    plt.show()

plot(res)

Det finnes imidlertid en enda bedre måte å visualisere ord-frekvenser på - ved å bruke **Word Cloud**. Vi må installere et annet bibliotek for å tegne ordskyen fra vår nøkkelordliste.


In [ ]:
!{sys.executable} -m pip install wordcloud

`WordCloud`-objektet har ansvar for å ta inn enten originaltekst eller forhåndsberegnet liste over ord med deres frekvenser, og returnerer et bilde som deretter kan vises ved hjelp av `matplotlib`:


In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

wc = WordCloud(background_color='white',width=800,height=600)
plt.figure(figsize=(15,7))
plt.imshow(wc.generate_from_frequencies({ k:v for k,v in res }))

Vi kan også sende inn den originale teksten til `WordCloud` - la oss se om vi klarer å få et tilsvarende resultat:


In [ ]:
plt.figure(figsize=(15,7))
plt.imshow(wc.generate(text))

In [ ]:
wc.generate(text).to_file('images/ds_wordcloud.png')

Du kan se at ordskyen nå ser mer imponerende ut, men den inneholder også mye støy (f.eks. urelaterte ord som `Retrieved on`). Vi får også færre nøkkelord som består av to ord, som *data scientist* eller *computer science*. Dette er fordi RAKE-algoritmen gjør en mye bedre jobb med å velge gode nøkkelord fra teksten. Dette eksempelet illustrerer viktigheten av dataforbehandling og rensing, fordi et klart bilde til slutt vil gjøre oss i stand til å ta bedre beslutninger.

I denne øvelsen har vi gått gjennom en enkel prosess for å trekke ut noe mening fra Wikipedia-tekst, i form av nøkkelord og ordsky. Dette eksempelet er ganske enkelt, men det demonstrerer godt alle typiske steg en dataforsker vil ta når de jobber med data, fra dataanskaffelse til visualisering.

I kurset vårt vil vi diskutere alle disse stegene i detalj. 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:
Dette dokumentet er oversatt ved hjelp av AI-oversettelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selv om vi streber etter nøyaktighet, vær oppmerksom på at automatiske oversettelser kan inneholde feil eller unøyaktigheter. Det opprinnelige dokumentet på originalspråket skal betraktes som den autoritative kilden. For kritisk informasjon anbefales profesjonell menneskelig oversettelse. Vi er ikke ansvarlige for eventuelle misforståelser eller feiltolkninger som oppstår ved bruk av denne oversettelsen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
